**<mark>Load Libraries and packages require for the execution**</mark>

In [73]:
# 📦 Required Imports & Session Setup

import concurrent.futures
import requests
from datetime import datetime
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, FloatType, NullType
)

# Initialize Spark session
spark = SparkSession.builder.getOrCreate()

StatementMeta(, 7a2305ae-152d-427a-8101-50d788c2cbeb, 75, Finished, Available, Finished)

In [ ]:
def get_access_token(resource: str = "https://database.windows.net/", name: str = "") -> str:
    """
    Fetch an AAD access token for the given resource (audience).
    Uses mssparkutils.credentials.getToken under the hood.
    """
    return mssparkutils.credentials.getToken(resource, name)


### 📖 Read SQL Data via JDBC with Token Authentication

This function executes a SQL query on a JDBC endpoint using token-based auth and returns the result as a Pandas DataFrame.


In [ ]:
def read_sql_data(query: str, jdbc_url: str, access_token: str) -> pd.DataFrame:
    """
    Execute a SQL query via JDBC using token authentication and return the results as a Pandas DataFrame.
    """
    # Prepare Java connection properties
    props = spark._sc._gateway.jvm.java.util.Properties()
    props.setProperty("accessToken", access_token)
    props.setProperty("encrypt", "true")

    # Acquire connection and statement
    driver_manager = spark._sc._gateway.jvm.java.sql.DriverManager
    con = driver_manager.getConnection(jdbc_url, props)
    stmt = con.prepareCall(query)

    data = []
    try:
        stmt.execute()
        rs = stmt.getResultSet()
        if rs:
            md = rs.getMetaData()
            col_count = md.getColumnCount()
            while rs.next():
                # Build row as dict
                row = {
                    md.getColumnName(i): rs.getString(i)
                    for i in range(1, col_count + 1)
                }
                data.append(row)
    finally:
        # Clean up resources
        try:
            stmt.close()
        except Exception as e:
            print(f"Warning: error closing statement: {e}")
        try:
            con.close()
        except Exception as e:
            print(f"Warning: error closing connection: {e}")

    # Convert to Pandas DataFrame and return
    return pd.DataFrame(data)

### 📝 Write Data to SQL Table via JDBC

This function `write_sql_data` appends a Spark DataFrame (or Pandas DataFrame) to a SQL table using JDBC with Azure AD token authentication.

In [1]:
def write_sql_data(df, table_name: str, jdbc_url: str, access_token: str) -> None:
    """
    Write / append a Spark or Pandas DataFrame to a SQL table via JDBC, using AAD token authentication.
    """
    # Convert Pandas to Spark DF if needed
    import pandas as pd
    spark_df = spark.createDataFrame(df) if isinstance(df, pd.DataFrame) else df

    try:
        print(f"[write_sql_data] Writing data to {table_name} via {jdbc_url}")
        spark_df.write \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("dbtable", table_name) \
            .option("accessToken", access_token) \
            .option("encrypt", "true") \
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
            .mode("append") \
            .save()
        print(f"[write_sql_data] Successfully wrote to {table_name}")
    except Exception as e:
        print(f"[write_sql_data] Error writing to {table_name}: {e}")
        # Optionally rethrow or wrap
        # raise

StatementMeta(, , -1, SessionStarting, , SessionStarting)

### Execute SQL Queries Concurrently via JDBC

This function executes multiple SQL queries in parallel (using threads) against a JDBC endpoint, capturing execution status, timing, and environment metadata.

In [ ]:
from datetime import datetime
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Tuple

def execute_sql_queries_concurrently(df, jdbc_url: str, concurrency: int = 5):
    """
    Execute SQL queries concurrently against a JDBC endpoint using token-based auth.
    Returns a Spark DataFrame containing:
      - QueryId, TrackName, QueryText
      - Status (“Success” or error message)
      - StartTime, EndTime, ExecutionTime (seconds)
      - Environment (Azure SQL or Fabric)
      - Concurrency (number of parallel threads)
    """
    access_token = get_access_token()
    environment = "Azure SQL" if "database.windows.net" in jdbc_url else "Fabric"

    def run_query(query_id, trackName, query_text) -> Tuple:
        try:
            props = spark._sc._gateway.jvm.java.util.Properties()
            props.setProperty("accessToken", access_token)
            props.setProperty("encrypt", "true")
            props.setProperty("trustServerCertificate", "true")
            driver_manager = spark._sc._gateway.jvm.java.sql.DriverManager

            start_time = datetime.utcnow()
            conn = driver_manager.getConnection(jdbc_url, props)
            stmt = conn.createStatement()
            stmt.execute(query_text)
            stmt.close()
            conn.close()
            end_time = datetime.utcnow()

            duration = (end_time - start_time).total_seconds()
            return (query_id, trackName, query_text, "Success",
                    start_time, end_time, duration)

        except Exception as e:
            logging.error("Error running query %s: %s", query_id, str(e))
            return (query_id, trackName, query_text, f"Error: {str(e)}",
                    None, None, None)

    # Convert Spark DF to Pandas for iteration
    query_pdf = df.select("QueryId", "TrackName", "QueryText").toPandas()

    results = []
    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = [
            executor.submit(run_query, row["QueryId"], row["TrackName"], row["QueryText"])
            for _, row in query_pdf.iterrows()
        ]
        for future in as_completed(futures):
            results.append(future.result())

    # Build result DataFrame
    result_schema = ["QueryId", "TrackName", "QueryText", "Status",
                     "StartTime", "EndTime", "ExecutionTime"]
    final_df = spark.createDataFrame(results, schema=result_schema)

    return (final_df
            .withColumn("Environment", lit(environment))
            .withColumn("Concurrency", lit(concurrency)))

In [ ]:
#Escapes single quotes in SQL string literals to prevent syntax errors or SQL injection issues.
def _escape_sql_literal(val: str) -> str:
    """Escape single quotes within SQL string literals by doubling them."""
    if val is None:
        return ""
    return str(val).replace("'", "''")

In [ ]:
def execute_sql_statements_with_token(
    sql_statements: List[str],
    jdbc_url: str,
    access_token: Optional[str] = None
) -> List[Dict[str, Optional[str]]]:
    """
    Execute one or more raw T-SQL statements via JDBC using an access token.
    Returns a list of dictionaries for each statement with keys:
      - "sql": the statement executed
      - "status": "success" or "failed"
      - "error": error message if failed, else None
    """
    results: List[Dict[str, Optional[str]]] = []
    if access_token is None:
        access_token = mssparkutils.credentials.getToken("pbi")

    conn = None
    try:
        # Setup connection properties
        props = spark._sc._gateway.jvm.java.util.Properties()
        props.setProperty("accessToken", access_token)
        props.setProperty("encrypt", "true")

        driver_manager = spark._sc._gateway.jvm.java.sql.DriverManager
        conn = driver_manager.getConnection(jdbc_url, props)

        for sql in sql_statements:
            try:
                stmt = conn.createStatement()
                stmt.execute(sql)
                results.append({"sql": sql, "status": "success", "error": None})
            except Exception as ex_stmt:
                results.append({"sql": sql, "status": "failed", "error": str(ex_stmt)})
            finally:
                try:
                    stmt.close()
                except Exception:
                    # Log or ignore close errors
                    pass

    except Exception as ex_conn:
        # If connection establishment fails
        err_msg = f"Connection setup failed: {ex_conn}"
        # Append an entry for each statement? Or a single global error?
        results.append({"sql": None, "status": "failed", "error": err_msg})
    finally:
        if conn:
            try:
                conn.close()
            except Exception:
                # Log or ignore
                pass

    return results

In [ ]:
def batch_insert_conn_if_not_exists(conn_list: list[dict], jdbc_url: str, exec_fn) -> list[dict]:
    """
    Loop over a list of connection dicts, build & execute SQL with “NOT EXISTS” logic.
    exec_fn(sql: str) => executes the SQL (e.g. via JDBC or your token-based execution).
    Returns a list of result dicts for each attempted insert.
    """
    results = []
    for conn in conn_list:
        try:
            sql = build_insert_if_not_exists_sql(conn)
        except Exception as e:
            results.append({
                "conn": conn,
                "status": "failed",
                "error": f"SQL build error: {e}"
            })
            continue

        try:
            resp = exec_fn(sql)
            results.append({
                "conn": conn,
                "status": "success",
                "response": resp
            })
        except Exception as ex:
            results.append({
                "conn": conn,
                "status": "failed",
                "error": str(ex),
                "sql": sql
            })
    return results

In [ ]:
def build_insert_if_not_exists_sql(conn: dict) -> str:
    """
    Given a connection-details dict, build a SQL INSERT … WHERE NOT EXISTS statement.
    """
    required = [
        "TrackName", "DatabaseName", "SourceServerName",
        "SourceDatabaseName", "TargetServerName",
        "TargetDatabaseName", "isActive"
    ]
    missing = [k for k in required if k not in conn]
    if missing:
        raise ValueError(f"Missing required keys: {missing}")

    # Escape each string field
    tn = _escape_sql_literal(conn["TrackName"])
    im_db = _escape_sql_literal(conn["DatabaseName"])
    im_ssn = _escape_sql_literal(conn["SourceServerName"])
    im_sdn = _escape_sql_literal(conn["SourceDatabaseName"])
    im_tsn = _escape_sql_literal(conn["TargetServerName"])
    im_tdn = _escape_sql_literal(conn["TargetDatabaseName"])
    is_act = 1 if conn["isActive"] else 0

    sql = f"""
    INSERT INTO metadata.tbl_Reconciliation_ConnectionDetails
        (TrackName,
         DatabaseName,
         SourceServerName,
         SourceDatabaseName,
         TargetServerName,
         TargetDatabaseName,
         isActive)
    SELECT
        '{tn}' AS TrackName,
        '{im_db}' AS DatabaseName,
        '{im_ssn}' AS SourceServerName,
        '{im_sdn}' AS SourceDatabaseName,
        '{im_tsn}' AS TargetServerName,
        '{im_tdn}' AS TargetDatabaseName,
        {is_act} AS isActive
    WHERE NOT EXISTS (
        SELECT 1 FROM metadata.tbl_Reconciliation_ConnectionDetails AS t
        WHERE LOWER(t.TrackName) = LOWER('{tn}') and LOWER(t.DatabaseName) = LOWER('{im_db}')
    );
    """
    return sql

### 🧾 Execute Multiple SQL Statements via JDBC (Token Auth)

This function executes a list of SQL statements sequentially against a JDBC endpoint using Azure AD token authentication. It returns a list of dicts capturing success/failure, timing, and error details.

In [ ]:
def execute_sql_statements(
    statements: list[str],
    jdbc_url: str,
    access_token: str | None = None
) -> list[dict]:
    """
    Execute each SQL statement via JDBC using token authentication.
    Returns a list of dicts with:
      - 'sql': the SQL statement
      - 'status': 'success' or 'failed'
      - 'start', 'end': timestamps (UTC) when execution began & ended
      - 'error': error message when failed, else None
    """
    if access_token is None:
        access_token = mssparkutils.credentials.getToken("pbi")

    conn = None
    stmt = None
    results = []
    try:
        # Prepare JDBC properties for token authentication
        props = spark._sc._gateway.jvm.java.util.Properties()
        props.setProperty("accessToken", access_token)
        props.setProperty("encrypt", "true")

        driver_manager = spark._sc._gateway.jvm.java.sql.DriverManager
        conn = driver_manager.getConnection(jdbc_url, props)
        stmt = conn.createStatement()

        for sql in statements:
            try:
                print(f"[execute_sql_statements] Executing: {sql}")
                ts_start = datetime.utcnow()
                stmt.execute(sql)
                ts_end = datetime.utcnow()
                results.append({
                    "sql": sql,
                    "status": "success",
                    "start": ts_start,
                    "end": ts_end,
                    "error": None
                })
            except Exception as ex_stmt:
                print(f"[execute_sql_statements] Failed: {sql}\nException: {ex_stmt}")
                results.append({
                    "sql": sql,
                    "status": "failed",
                    "start": None,
                    "end": None,
                    "error": str(ex_stmt)
                })
                # continue to next statement
    except Exception as ex_conn:
        # Connection-level failure
        raise RuntimeError(f"Failed to open JDBC connection or statement: {ex_conn}") from ex_conn
    finally:
        # Clean up resources
        try:
            if stmt:
                stmt.close()
        except Exception as close_ex:
            print(f"[execute_sql_statements] Warning: could not close stmt: {close_ex}")
        try:
            if conn:
                conn.close()
        except Exception as close_ex2:
            print(f"[execute_sql_statements] Warning: could not close connection: {close_ex2}")

    return results

### 📋 get_tables  
Fetches table/view metadata (schema, name, type) excluding system schemas and internal tables. Returns a Spark DataFrame or `None` on failure.


In [ ]:
def get_tables(jdbc_url: str, access_token: str):
    """
    Fetch tables and views from the SQL endpoint, skipping certain schemas/tables.
    Returns a Spark DataFrame, or None if an error occurs.
    """
    query = """
    SELECT TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_SCHEMA NOT IN ('sys', 'queryinsights')
      AND TABLE_NAME NOT IN ('CssLogTable', 'FactPSIQA')
    """
    try:
        df = (
            spark.read
                 .format("jdbc")
                 .option("url", jdbc_url)
                 .option("dbtable", f"({query}) AS query_alias")
                 .option("accessToken", access_token)
                 .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
                 .load()
        )
        return df
    except Exception as e:
        print(f"[get_tables] Error occurred: {e}")
        return None

### 🧾 Fetch Column Metadata (Excluding System / Internal)

This function retrieves column-level metadata (schema, table, column name, data type, nullability) from a SQL endpoint, excluding system schemas and internal tables. Returns a Spark DataFrame or `None` on error.


In [ ]:
def get_columns(jdbc_url: str, access_token: str):
    """
    Fetch column definitions (name, type, nullability) excluding certain schemas/tables.
    Returns a Spark DataFrame or None on error.
    """
    query = """
    SELECT
        TABLE_SCHEMA,
        TABLE_NAME,
        COLUMN_NAME,
        DATA_TYPE,
        IS_NULLABLE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA NOT IN ('sys', 'queryinsights')
      AND TABLE_NAME NOT IN ('CssLogTable')
    """
    try:
        df = (
            spark.read
                 .format("jdbc")
                 .option("url", jdbc_url)
                 .option("dbtable", f"({query}) AS query_alias")
                 .option("accessToken", access_token)
                 .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
                 .load()
        )
        return df
    except Exception as e:
        print(f"[get_columns] Error: {str(e)}")
        return None


In [ ]:
def get_record_count(jdbc_url: str, access_token: str, schema: str, table: str) -> int | None:
    """
    Return the total number of records in the specified schema.table.
    Returns an integer, or None if an error occurs.
    """
    query = f"SELECT COUNT(*) AS CountVal FROM [{schema}].[{table}]"
    try:
        df = (
            spark.read
                 .format("jdbc")
                 .option("url", jdbc_url)
                 .option("dbtable", f"({query}) AS query_alias")
                 .option("accessToken", access_token)
                 .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
                 .load()
        )
        # collect the result and return as int
        count_val = df.collect()[0][0]
        return int(count_val)
    except Exception as e:
        print(f"[get_record_count] Error for {schema}.{table}: {e}")
        return None

### 📤 `writeSQLData` — Append Data to SQL via JDBC with Token Auth

Writes a Pandas or Spark DataFrame to a SQL table via JDBC, using Azure AD token authentication.  
By default, it appends the data (does not overwrite).


In [ ]:
def writeSQLData(df, table_name: str, jdbc_url: str, access_token: str = None):
    """
    Write a DataFrame (Pandas or Spark) to a SQL table via JDBC using token-based authentication.
    """
    import pandas as pd

    if access_token is None:
        access_token = mssparkutils.credentials.getToken("pbi")

    try:
        # Convert Pandas to Spark DF if needed
        spark_df = spark.createDataFrame(df) if isinstance(df, pd.DataFrame) else df

        # Optionally skip if DataFrame is empty
        row_count = spark_df.count()
        if row_count == 0:
            print(f"[writeSQLData] No rows to write to {table_name}.")
            return

        print(f"[writeSQLData] Writing {row_count} rows to {table_name}")

        spark_df.write \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("dbtable", table_name) \
            .option("accessToken", access_token) \
            .option("encrypt", "true") \
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
            .mode("append") \
            .save()

        print(f"[writeSQLData] Successfully wrote to {table_name}")

    except Exception as e:
        print(f"[writeSQLData] Error occurred while writing to {table_name}: {e}")

In [ ]:
def get_record_count_safe(jdbc_url: str, token: str, schema: str, table: str) -> int | str:
    """
    Safely retrieve row count for a given schema.table.
    Returns the integer count, or an error message string on failure.
    """
    try:
        return get_record_count(jdbc_url, token, schema, table)
    except Exception as e:
        err_msg = f"Error for {schema}.{table}: {e}"
        print(f"[get_record_count_safe] {err_msg}")
        return err_msg

## 🔍 Parallel Record Counts Between Prod & Fabric

Given a list of `(schema, table)` pairs, this function runs `get_record_count` concurrently (via thread pooling) against both production and fabric JDBC endpoints, and returns a Pandas DataFrame comparing the counts (or error messages).


In [ ]:
def parallel_record_counts(
    common_tables: list[tuple[str, str]],
    prod_jdbc_url: str,
    prod_token: str,
    fabric_jdbc_url: str,
    fabric_token: str,
    max_workers: int = 10
) -> pd.DataFrame:
    """
    Fetch row counts in parallel for given (schema, table) pairs from both prod and fabric endpoints.

    Returns a Pandas DataFrame with columns:
      - TABLE_SCHEMA
      - TABLE_NAME
      - count_prod
      - count_fabric
    If an error occurs for a given table, the count is a string with the error message.
    """

    def task(schema_table: tuple[str, str]) -> dict:
        schema, table = schema_table
        try:
            cnt_prod = get_record_count(prod_jdbc_url, prod_token, schema, table)
        except Exception as e:
            cnt_prod = f"Error: {str(e)}"
        try:
            cnt_fab = get_record_count(fabric_jdbc_url, fabric_token, schema, table)
        except Exception as e:
            cnt_fab = f"Error: {str(e)}"
        return {
            "TABLE_SCHEMA": schema,
            "TABLE_NAME": table,
            "count_prod": cnt_prod,
            "count_fabric": cnt_fab
        }

    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            results = list(executor.map(task, common_tables))
        return pd.DataFrame(results)
    except Exception as e:
        print(f"[parallel_record_counts] Error during parallel execution: {str(e)}")
        # Return empty DataFrame with expected columns on failure
        return pd.DataFrame(columns=["TABLE_SCHEMA", "TABLE_NAME", "count_prod", "count_fabric"])

### 📉 Get Null and Total Counts for a Column

This function returns a tuple `(null_count, total_count)` for a specified column in a schema.table. In case of error, it returns `(None, None)`.


In [ ]:
def get_null_count(jdbc_url: str, access_token: str, schema: str, table: str, column: str) -> tuple[int | None, int | None]:
    """
    Return (null_count, total_count) for a given column in schema.table.
    Returns a pair of integers or (None, None) if an error occurs.
    """
    query = (
        f"SELECT "
        f"SUM(CASE WHEN [{column}] IS NULL THEN 1 ELSE 0 END) AS NullCount, "
        f"COUNT(*) AS TotalCount "
        f"FROM [{schema}].[{table}]"
    )
    try:
        df = (
            spark.read
                 .format("jdbc")
                 .option("url", jdbc_url)
                 .option("dbtable", f"({query}) AS query_alias")
                 .option("accessToken", access_token)
                 .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
                 .load()
        )
        row = df.collect()[0]
        null_cnt = row[0]
        total_cnt = row[1]
        # Cast to int if not None
        return (int(null_cnt) if null_cnt is not None else None,
                int(total_cnt) if total_cnt is not None else None)
    except Exception as e:
        print(f"[get_null_count] Error for {schema}.{table}.{column}: {e}")
        return (None, None)

## 🛠️ Main Reconciliation Logic

This function orchestrates three comparison phases between the **prod** and **fabric** databases:

1. **Table existence diff**  
2. **Schema / column differences**  
3. **Record count comparisons**

It logs the results (via `writeSQLData`) into reconciliation tables and prints / displays interim dataframes for review.

In [ ]:
def main(
    prod_jdbc_url: str,
    prod_token: str,
    fabric_jdbc_url: str,
    fabric_token: str,
    trackName: str,
    databaseName: str,
    config_jdbc_url: str
):
    def create_df_source(pairs: list[tuple], source_label: str):
        """Create a Spark DataFrame for table diff, marking origin side."""
        df = spark.createDataFrame(pairs, schema=["TABLE_SCHEMA", "TABLE_NAME"])
        return (df
                .withColumn("COMPARE_RESULT", lit(source_label))
                .withColumn("databaseName", lit(databaseName))
                .withColumn("trackName", lit(trackName)))

    # 1. Compare table existence
    try:
        tables_prod = get_tables(prod_jdbc_url, prod_token)
        tables_fabric = get_tables(fabric_jdbc_url, fabric_token)

        set_prod = set(tables_prod.select("TABLE_SCHEMA", "TABLE_NAME").collect())
        set_fab = set(tables_fabric.select("TABLE_SCHEMA", "TABLE_NAME").collect())

        only_in_prod = list(set_prod - set_fab)
        only_in_fabric = list(set_fab - set_prod)

        df_prod_only = create_df_source(only_in_prod, "only_in_prod")
        df_fab_only = create_df_source(only_in_fabric, "only_in_fabric")

        union_df = df_prod_only.union(df_fab_only) \
                    .withColumn("executionDateTime", current_timestamp())

        display(union_df)
        writeSQLData(union_df.toPandas(), "logging.tbl_Reconciliation_TableDiff_Current", config_jdbc_url)

    except Exception as e:
        print(f"[main] Error during table diff: {e}")

    # 2. Compare schemas / columns
    try:
        cols_prod = get_columns(prod_jdbc_url, prod_token)
        cols_fab = get_columns(fabric_jdbc_url, fabric_token)

        pdf = cols_prod.toPandas()
        fdf = cols_fab.toPandas()

        merged = pdf.merge(
            fdf,
            on=["TABLE_SCHEMA", "TABLE_NAME", "COLUMN_NAME"],
            how="outer",
            suffixes=("_Prod", "_Fabric"),
            indicator=True
        )

        # Keep only columns present in both sides
        schema_diff = merged[merged["_merge"] == "both"]

        spark_schema_df = spark.createDataFrame(schema_diff)

        # Drop columns whose data types became NullType
        null_fields = [fld.name for fld in spark_schema_df.schema.fields if isinstance(fld.dataType, NullType)]
        if null_fields:
            spark_schema_df = spark_schema_df.drop(*null_fields)

        spark_schema_df = (spark_schema_df
                           .withColumn("executionDateTime", current_timestamp())
                           .withColumn("trackName", lit(trackName))
                           .withColumn("databaseName", lit(databaseName)))

        display(spark_schema_df)
        writeSQLData(spark_schema_df.toPandas(), "logging.tbl_Reconciliation_SchemaDiff_Current", config_jdbc_url)

    except Exception as e:
        print(f"[main] Error during schema diff: {e}")

    # 3. Compare record counts on common tables
    try:
        common_tables = list(set_prod & set_fab)
        rec_counts_df = parallel_record_counts(common_tables,
                                               prod_jdbc_url, prod_token,
                                               fabric_jdbc_url, fabric_token)

        rec_spark = spark.createDataFrame(rec_counts_df)

        null_fields = [fld.name for fld in rec_spark.schema.fields if isinstance(fld.dataType, NullType)]
        if null_fields:
            rec_spark = rec_spark.drop(*null_fields)

        rec_spark = (rec_spark
                     .withColumn("executionDateTime", current_timestamp())
                     .withColumn("trackName", lit(trackName))
                     .withColumn("databaseName", lit(databaseName)))

        display(rec_spark)
        writeSQLData(rec_spark.toPandas(), "logging.tbl_Reconciliation_RecordCountsDiff_Current", config_jdbc_url)

    except Exception as e:
        print(f"[main] Error during record count diff: {e}")